# 使用 Trainer API 或者 Keras 微调一个模型

Install the Transformers, Datasets, and Evaluate libraries to run this notebook.

In [ ]:
!pip install datasets evaluate transformers[sentencepiece]

🤗 Transformers 提供了一个 `Trainer` 类，可以帮助你**在数据集上微调任何预训练模型**。

在上一节中完成所有数据预处理工作后，你只需完成几个步骤来定义 Trainer 。最困难的部分可能是准备运行` Trainer.train() `所需的环境，因为在 CPU 上运行速度非常慢。

In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer, DataCollatorWithPadding

raw_datasets = load_dataset("glue", "mrpc")
checkpoint = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)


def tokenize_function(example):
    return tokenizer(example["sentence1"], example["sentence2"], truncation=True)


tokenized_datasets = raw_datasets.map(tokenize_function, batched=True)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

##Training

在我们定义 Trainer 之前第一步要定义一个 `TrainingArguments` 类，它**包含 Trainer 在训练和评估中使用的所有超参数**。你只需要提供的参数是一个用于保存训练后的模型以及训练过程中的 checkpoint 的**目录**。对于其余的参数你可以保留默认值，这对于简单的微调应该效果就很好了。

In [ ]:
from transformers import TrainingArguments

#test-trainer是目录
training_args = TrainingArguments("test-trainer")

二步是定义我们的模型。我们将使用 `AutoModelForSequenceClassification` 类，它有两个参数：

In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2)

一旦有了我们的模型，我们就可以定义一个 **Trainer** 把到目前为止构建的所有对象 —— `model` ，`training_args`，`训练和验证数据集`， `data_collator` 和 `tokenizer` 传递给 **Trainer** ：

In [ ]:
from transformers import Trainer

trainer = Trainer(
    model,
    training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    tokenizer=tokenizer,
)

In [ ]:
trainer.train()

开始微调，每 500 步报告一次训练损失。然而它不会告诉你模型的性能（或质量）如何。这是因为：

*  我们没有告诉 Trainer 在训练过程中进行评估。
*  我们没有为 Trainer 提供一个 compute_metrics() 函数来计算上述评估过程的指标（否则评估将只会输出 loss，但这不是一个非常直观的数字）。

##评估

让我们看看如何构建一个有用的 `compute_metrics()` 函数，并在下次训练时使用它。该函数必须接收一个 **EvalPrediction** 对象（它是一个带有 predictions 和 label_ids 字段的参数元组），并将返回一个字符串映射到浮点数的字典（字符串是返回的指标名称，而浮点数是其值）。

为了从我们的模型中获得预测结果，可以使用 `Trainer.predict()` 命令：



In [ ]:
predictions = trainer.predict(tokenized_datasets["validation"])
print(predictions.predictions.shape, predictions.label_ids.shape)

(408, 2) (408,)

调用 Trainer 的预测接口，在验证集上执行推理，返回一个命名元组对象**EvalPrediction**，包含三大属性：

*  `.predictions`：模型输出logits（原始得分，未经过 softmax）
*  `.label_ids`：样本真实标签
*  `.metrics`：推理自动算出的损失、耗时等指标，当自定义 compute_metrics() 时包含其返回的结果。


**predictions.predictions.shape**
*  形状：[样本总数, num_labels]
*  本例 MRPC 二分类，就是[408, 2]，代表验证集 408 条样本，每个样本输出 2 个类别的 logit 值。

**predictions.label_ids.shape**
*  形状：[样本总数]
*  一维数组，存储每条样本真实标签（0 或 1）。

**logits**

*  定义：模型最后一层输出未经过激活函数（softmax/sigmoid）的原始分值，就是 logits。值域没有限制，可正可负。
*  `np.argmax(logits, axis=-1)`：转化为可以与我们的标签进行比较的预测值，选出分值更大的那一列索引，作为预测类别。



In [ ]:
import numpy as np

# predictions.predictions的输出是logits
preds = np.argmax(predictions.predictions, axis=-1)

我们现在可以将这些 preds 与标签进行比较。为了构建我们的 **compute_metric()** 函数，使用 `evaluate.load()` 函数。返回的对象有一个 **`compute()` 方法**，我们可以用它来进行指标的计算：

In [ ]:
import evaluate

metric = evaluate.load("glue", "mrpc")
metric.compute(predictions=preds, references=predictions.label_ids)

{'accuracy': 0.8578431372549019, 'f1': 0.8996539792387542}

最后把所有东西打包在一起，我们就得到了 **`compute_metrics()` 函数**：

In [ ]:
def compute_metrics(eval_preds):
    metric = evaluate.load("glue", "mrpc")
    logits, labels = eval_preds
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

为了查看模型在每个训练周期结束时的好坏，我们使用 compute_metrics() 函数定义一个新的 Trainer：

In [ ]:
training_args = TrainingArguments("test-trainer", evaluation_strategy="epoch")
model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2)

trainer = Trainer(
    model,
    training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    tokenizer=tokenizer,
    #新增了我们自定义的函数
    compute_metrics=compute_metrics,
)

执行，它将在每个 epoch 结束时在训练损失的基础上报告验证损失和指标。

In [ ]:
trainer.train()